# Calculate IST for all offensive players

In [26]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio


In [ ]:
import sys
from pathlib import Path
# Add project root to path for imports
ROOT = Path.cwd().parent  
sys.path.append(str(ROOT))


In [28]:
df = pd.read_parquet("../data/processed/def_features_test/defense_01.18.2016.GSW.at.CLE_21500622.parquet")


In [29]:
df['SHOT_MADE_FLAG'].value_counts()

SHOT_MADE_FLAG
0    57
1    53
Name: count, dtype: int64

In [33]:
def calculate_ist_for_row(row):
    """
    Calculates the IST for all 5 offensive players across all frames.
    Returns a NumPy array of shape (num_frames, 5).
    """
    # 1. Stack the trajectories into 2D arrays of shape (frames, 5)
    off_x = np.column_stack([row[f"off{i}_x_traj"] for i in range(1, 6)])
    off_y = np.column_stack([row[f"off{i}_y_traj"] for i in range(1, 6)])
    off_q = np.column_stack([row[f"off{i}_q_traj"] for i in range(1, 6)])
    
    def_x = np.column_stack([row[f"def{i}_x_traj"] for i in range(1, 6)])
    def_y = np.column_stack([row[f"def{i}_y_traj"] for i in range(1, 6)])
    
    # 2. Compute Distances using Broadcasting
    # We add axes to compare every offensive player to every defender
    # off shape: (frames, 5, 1) | def shape: (frames, 1, 5)
    dx = off_x[:, :, np.newaxis] - def_x[:, np.newaxis, :]
    dy = off_y[:, :, np.newaxis] - def_y[:, np.newaxis, :]
    
    # Distance matrix shape: (frames, 5 offense, 5 defense)
    distances = np.sqrt(dx**2 + dy**2)
    
    # 3. Find Nearest Defender (Openness)
    # Take the minimum distance across the defender axis. Shape: (frames, 5)
    nearest_def_dist = np.min(distances, axis=2)
    
    # Define O (Openness). If your model uses raw distance to nearest defender:
    O = nearest_def_dist 
    
    # (Optional) If your model caps Openness at a certain distance (e.g., 15 feet)
    # O = np.clip(nearest_def_dist, 0, 15)
    
    # 4. Calculate IST for all 5 players at all frames
    # Applies the exponents matrix-wide instantly
    ist_matrix = (off_q ** 2.16) * (O ** 1.03)
    
    return ist_matrix

In [34]:
df.columns

Index(['GAME_ID', 'SHOT_EVENT_ID', 'tracking_event_id', 'event_list_idx',
       'PERIOD', 'game_clock', 'PLAYER_ID', 'TEAM_ID', 'flipped_coordinates',
       'ball_x_traj', 'ball_y_traj', 'off1_pid', 'off1_x_traj', 'off1_y_traj',
       'off1_q_traj', 'def1_pid', 'def1_x_traj', 'def1_y_traj', 'off2_pid',
       'off2_x_traj', 'off2_y_traj', 'off2_q_traj', 'def2_pid', 'def2_x_traj',
       'def2_y_traj', 'off3_pid', 'off3_x_traj', 'off3_y_traj', 'off3_q_traj',
       'def3_pid', 'def3_x_traj', 'def3_y_traj', 'off4_pid', 'off4_x_traj',
       'off4_y_traj', 'off4_q_traj', 'def4_pid', 'def4_x_traj', 'def4_y_traj',
       'off5_pid', 'off5_x_traj', 'off5_y_traj', 'off5_q_traj', 'def5_pid',
       'def5_x_traj', 'def5_y_traj', 'release_frame_global_idx',
       'pbp_frame_global_idx', 'local_release_idx', 'local_pbp_idx',
       'ball_z_traj', 'SHOT_MADE_FLAG'],
      dtype='object')

In [36]:
def get_court_shapes_plotly():
    """Returns standard NBA court lines for Plotly."""
    shapes = [
        dict(type="rect", x0=0, y0=0, x1=94, y1=50, line=dict(color="black", width=2)),
        dict(type="line", x0=47, y0=0, x1=47, y1=50, line=dict(color="black", width=2)),
        dict(type="circle", x0=4.5, y0=24.25, x1=6, y1=25.75, line=dict(color="black", width=2)),
        dict(type="circle", x0=94-6, y0=24.25, x1=94-4.5, y1=25.75, line=dict(color="black", width=2))
    ]
    r = 23.75
    shapes.extend([
        dict(type="path", path=f"M {r+5.25},47.5 A {r},{r} 0 0,0 {r+5.25},2.5", line=dict(color="black", width=2)),
        dict(type="path", path=f"M {94-(r+5.25)},47.5 A {r},{r} 0 0,1 {94-(r+5.25)},2.5", line=dict(color="black", width=2)),
        dict(type="line", x0=5.25, y0=2.5, x1=5.25+r, y1=2.5, line=dict(color="black", width=2)),
        dict(type="line", x0=5.25, y0=47.5, x1=5.25+r, y1=47.5, line=dict(color="black", width=2)),
        dict(type="line", x0=94-5.25, y0=2.5, x1=94-(r+5.25), y1=2.5, line=dict(color="black", width=2)),
        dict(type="line", x0=94-5.25, y0=47.5, x1=94-(r+5.25), y1=47.5, line=dict(color="black", width=2))
    ])
    return shapes

In [44]:
def animate_single_play(row, d0=18.0, k=0.3, b_floor=0.4):
    """
    Takes a single row from your DataFrame (e.g., df.iloc[0]) 
    and returns an interactive Plotly animation of the play with IST scores.
    Now includes logistic distance-to-ball decay.
    """
    pio.renderers.default = "browser" # Pops open in your web browser
    
    # 1. Extract Trajectories
    num_frames = len(row['ball_x_traj'])
    
    off_traj = np.zeros((num_frames, 5, 2))
    def_traj = np.zeros((num_frames, 5, 2))
    q_traj = np.zeros((num_frames, 5))
    ball_traj = np.zeros((num_frames, 2))
    
    # Un-normalize coordinates back to 94x50
    for i in range(1, 6):
        off_traj[:, i-1, 0] = np.array(row[f'off{i}_y_traj']) + 5.25  # Court X
        off_traj[:, i-1, 1] = np.array(row[f'off{i}_x_traj']) + 25.0  # Court Y
        
        def_traj[:, i-1, 0] = np.array(row[f'def{i}_y_traj']) + 5.25
        def_traj[:, i-1, 1] = np.array(row[f'def{i}_x_traj']) + 25.0
        
        q_traj[:, i-1] = np.array(row[f'off{i}_q_traj'])
        
    ball_traj[:, 0] = np.array(row['ball_y_traj']) + 5.25
    ball_traj[:, 1] = np.array(row['ball_x_traj']) + 25.0

    # 2. Calculate IST (Now Including Ball Distance)
    
    # A. Defender Openness
    dx = off_traj[:, :, None, 0] - def_traj[:, None, :, 0]
    dy = off_traj[:, :, None, 1] - def_traj[:, None, :, 1]
    distances_to_def = np.sqrt(dx**2 + dy**2)
    openness = np.min(distances_to_def, axis=2) # Nearest defender
    
    # B. Distance to Ball
    # Broadcast ball_traj (frames, 2) -> (frames, 1, 2) to subtract from off_traj (frames, 5, 2)
    dx_ball = off_traj[:, :, 0] - ball_traj[:, None, 0]
    dy_ball = off_traj[:, :, 1] - ball_traj[:, None, 1]
    dist_to_ball = np.sqrt(dx_ball**2 + dy_ball**2)
    
   # Calculate raw decay, then compress it so it never drops below b_floor
    raw_logistic = 1.0 / (1.0 + np.exp(k * (dist_to_ball - d0)))
    b_factor = b_floor + (1.0 - b_floor) * raw_logistic
    
    # C. Final IST Calculation (Q * O * B)
    ist = (q_traj ** 2.16) * (openness ** 1.03) * b_factor

    # 3. Build Plotly Figure
    fig = go.Figure()

    def get_text_labels(ist_array):
        return [f"<b>{val:.2f}</b>" for val in ist_array]

    # Initial frame traces
    fig.add_trace(go.Scatter(x=def_traj[0, :, 0], y=def_traj[0, :, 1], mode='markers', 
                             marker=dict(color='blue', size=15), name='Defense'))
    fig.add_trace(go.Scatter(x=off_traj[0, :, 0], y=off_traj[0, :, 1], mode='markers+text', 
                             text=get_text_labels(ist[0]), textposition="top center", 
                             marker=dict(color='red', size=15), name='Offense'))
    fig.add_trace(go.Scatter(x=[ball_traj[0, 0]], y=[ball_traj[0, 1]], mode='markers', 
                             marker=dict(color='orange', size=12), name='Ball'))

    # Build animation frames
    frames = []
    for k in range(num_frames):
        frame_data = [
            go.Scatter(x=def_traj[k, :, 0], y=def_traj[k, :, 1]),
            go.Scatter(x=off_traj[k, :, 0], y=off_traj[k, :, 1], text=get_text_labels(ist[k])),
            go.Scatter(x=[ball_traj[k, 0]], y=[ball_traj[k, 1]])
        ]
        frames.append(go.Frame(data=frame_data, name=str(k)))
        
    fig.frames = frames

    # Animation layout and controls
    fig.update_layout(
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play", method="animate", 
                          args=[None, {"frame": {"duration": 50, "redraw": True}, 
                                       "fromcurrent": True, "transition": {"duration": 0}}])]
        )],
        sliders=[dict(
            steps=[dict(method='animate', 
                        args=[[f.name], dict(mode='immediate', frame=dict(duration=50, redraw=True), 
                                             transition=dict(duration=0))]) for f in fig.frames],
            transition=dict(duration=0), x=0, xanchor="left", len=1,
        )],
        title_text=f"Play Animation with Ball Factor IST",
        shapes=get_court_shapes_plotly(),
        xaxis=dict(range=[0, 94], autorange=False, showgrid=False, zeroline=False),
        yaxis=dict(range=[0, 50], autorange=False, scaleanchor="x", scaleratio=1, showgrid=False, zeroline=False),
        plot_bgcolor="white"
    )
    
    return fig

In [45]:
fig = animate_single_play(df.iloc[9])
fig.show()